In [21]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import sklearn
import scipy
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [2]:
data = np.random.rand(5, 10)  # 5 entities, each contains 10 features
label = np.random.randint(2, size=5)  # binary target

In [8]:
# Separate train/test data
features = ["WD_1_1_1", "NEE_PI_F"]

df = pd.read_csv("/Users/ktmall07/school/PhD/fluxcourse/FluxCourseProject/project/FluxCourseProject/data/tower/AMF_US-Syv_BASE_HH_33-5.csv")

In [16]:
df = df[features]
df = df[df['WD_1_1_1'] != -9999]

print(len(df))

301762


In [17]:
wd = df['WD_1_1_1']

conditions = [
    (wd >= 330) | (wd < 30),  # Category 1 (Wraps around 360/0)
    (wd >= 30) & (wd < 90),  # Category 2
    (wd >= 90) & (wd < 150),  # Category 3
    (wd >= 150) & (wd < 210),  # Category 4
    (wd >= 210) & (wd < 270),  # Category 5
    (wd >= 270) & (wd < 330),  # Category 6
]

# 2. Define the category labels
categories = [1, 2, 3, 4, 5, 6]

# 3. Apply the conditions to create a new column
df["wd_category"] = np.select(conditions, categories, default=np.nan)

In [18]:
df

,WD_1_1_1,NEE_PI_F,wd_category
10162,191.4370,-9999.000000,4.0
10164,190.4080,-9999.000000,4.0
10165,189.5960,-9999.000000,4.0
10166,191.4570,-9999.000000,4.0
10167,193.0780,-9999.000000,4.0
...,...,...,...
437999,128.4790,9.159056,3.0
438000,129.2706,6.659317,3.0
438001,129.7466,5.647617,3.0
438002,130.7000,4.499113,3.0


In [23]:
df_clean = df.copy()

df_clean["wd_category"] = df_clean["wd_category"].astype("str")

model = smf.glm(
    formula="NEE_PI_F ~ C(wd_category)", data=df_clean, family=sm.families.Gaussian()
).fit()

# Print the results summary
print(model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:               NEE_PI_F   No. Observations:               301762
Model:                            GLM   Df Residuals:                   301756
Model Family:                Gaussian   Df Model:                            5
Link Function:               Identity   Scale:                      8.6791e+06
Method:                          IRLS   Log-Likelihood:            -2.8387e+06
Date:                Thu, 18 Jun 2026   Deviance:                   2.6190e+12
Time:                        16:47:10   Pearson chi2:                 2.62e+12
No. Iterations:                     3   Pseudo R-squ. (CS):           0.001404
Covariance Type:            nonrobust                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept              -960.58

micromols per m^2 per s

In [25]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Run Tukey's Honest Significant Difference test
tukey = pairwise_tukeyhsd(
    endog=df_clean["NEE_PI_F"], groups=df_clean["wd_category"], alpha=0.05
)
print(tukey)

   Multiple Comparison of Means - Tukey HSD, FWER=0.05   
group1 group2  meandiff p-adj    lower     upper   reject
---------------------------------------------------------
   1.0    2.0   70.4365 0.0491    0.1507  140.7223   True
   1.0    3.0  144.3701    0.0   86.9617  201.7785   True
   1.0    4.0   54.4343 0.0419    1.1497  107.7188   True
   1.0    5.0  -10.0186 0.9946  -63.0155   42.9784  False
   1.0    6.0 -204.2115    0.0 -259.1363 -149.2868   True
   2.0    3.0   73.9336 0.0248    5.6537  142.2135   True
   2.0    4.0  -16.0022 0.9816  -80.8533   48.8489  False
   2.0    5.0   -80.455 0.0052 -145.0701    -15.84   True
   2.0    6.0  -274.648    0.0 -340.8535 -208.4426   True
   3.0    4.0  -89.9358    0.0  -140.545  -39.3266   True
   3.0    5.0 -154.3886    0.0  -204.695 -104.0823   True
   3.0    6.0 -348.5816    0.0  -400.915 -296.2482   True
   4.0    5.0  -64.4528 0.0008 -109.9967  -18.9089   True
   4.0    6.0 -258.6458    0.0 -306.4193 -210.8724   True
   5.0    6.0 